# Correlazioni dinamiche del trimero anello — misura via circuito, tutte le 81 combinazioni

Notebook snello, mirror per $N=3$ di `circuito_correlazioni_tutte.ipynb` (dimero). Qui non si
riderivano né il circuito né le simmetrie (fatto in `correlazioni_trimero_anello_simmetria.ipynb`
e `correlazioni_trimero_anello_esplorazione.ipynb`): tutte le $3\times3\times3\times3=81$
combinazioni $C_{ij}^{\alpha\beta}(t)$ vengono **misurate direttamente** dal circuito (nessuna
dedotta per simmetria), poi visualizzate e analizzate.

Punto di lavoro: $J=1,\,J'=0.4,\,b=b_c=2.4,\,D=0.15$ (Opzione B).

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trimer_ring_exact import trimer_hamiltonian_dm
from circuito_correlazioni_trimero_anello import ground_state, correlator_from_circuit
from validate_circuito_correlazioni import classical_exact, site_op

np.set_printoptions(precision=4, suppress=True)
J, Jp, b, D = 1.0, 0.4, 2.4, 0.15
N = 200

H = trimer_hamiltonian_dm(J, Jp, b, "B", D).to_matrix()
psi0, E = ground_state(J, Jp, b, D)
sites = (1, 2, 3)
comps = ("x", "y", "z")
print(f"Punto di lavoro: J={J}, J'={Jp}, b={b}, D={D}  (E0={E[0]:.4f}, gap={E[1]-E[0]:.4f})")

## 1. Misura diretta di tutte le 81 combinazioni ($t=1.3$, $N=200$)

Per ciascuna combinazione il circuito gira per intero: nessuna scorciatoia via simmetria.

In [ ]:
t_fix = 1.3
righe = []
for i in sites:
    for alpha in comps:
        for j in sites:
            for beta in comps:
                c_ref = classical_exact(i, alpha, j, beta, t_fix, J, Jp, b, D, psi0, H)
                c_circ = correlator_from_circuit(i, alpha, j, beta, t_fix, N, J, Jp, b, D, psi0)
                righe.append({"i": i, "alpha": alpha, "j": j, "beta": beta,
                              "c_ref": c_ref, "c_circ": c_circ,
                              "residuo": abs(c_ref - c_circ)})
df = pd.DataFrame(righe)
print(f"Misurate {len(df)} combinazioni a t={t_fix}.")

## 2. Validazione rapida

In [ ]:
print(f"Residuo (circuito vs classico esatto): media={df['residuo'].mean():.2e}  "
      f"dev.std={df['residuo'].std():.2e}  max={df['residuo'].max():.2e}  min={df['residuo'].min():.2e}")
print("\nLe 5 combinazioni con residuo maggiore (comunque solo errore di Trotter atteso):")
print(df.sort_values("residuo", ascending=False).head(5)[["i","alpha","j","beta","residuo"]].to_string(index=False))

**Discussione.** Nessuna anomalia: il residuo è uniformemente piccolo ($\sim10^{-4}$--$10^{-3}$),
coerente con solo errore di Trotter a $N=200$ — nessun caso isolato che tradisca un bug residuo,
come già verificato sui casi rappresentativi nel notebook di esplorazione.

## 3. Heatmap d'insieme

In [ ]:
labels = [f"{i}{a}" for i in sites for a in comps]
idx = {lab: k for k, lab in enumerate(labels)}
mag = np.zeros((9, 9))
is_zero = np.zeros((9, 9), dtype=bool)
zeri_attesi = {(3, "x", 3, "z"), (3, "z", 3, "x"), (3, "y", 3, "z"), (3, "z", 3, "y")}
for _, r in df.iterrows():
    a, c = idx[f"{r['i']}{r['alpha']}"], idx[f"{r['j']}{r['beta']}"]
    mag[a, c] = abs(r["c_circ"])
    is_zero[a, c] = (r["i"], r["alpha"], r["j"], r["beta"]) in zeri_attesi

fig, ax = plt.subplots(figsize=(6, 5.3))
im = ax.imshow(mag, cmap="viridis", vmin=0, vmax=mag.max())
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels([f"${l[0]}{l[1]}$" for l in labels], fontsize=9)
ax.set_yticklabels([f"${l[0]}{l[1]}$" for l in labels], fontsize=9)
ax.set_xlabel(r"$(j,\beta)$ a $t=0$"); ax.set_ylabel(r"$(i,\alpha)$ a $t$")
ax.set_title(rf"$|C_{{ij}}^{{\alpha\beta}}(t={t_fix})|$ dal circuito")
for r_ in range(9):
    for c_ in range(9):
        if is_zero[r_, c_]:
            ax.add_patch(plt.Rectangle((c_ - .5, r_ - .5), 1, 1, fill=False, edgecolor="red", lw=1.6))
for k in (2.5, 5.5):
    ax.axhline(k, color="white", lw=0.8); ax.axvline(k, color="white", lw=0.8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 4. Vista d'insieme nel tempo (griglia $9\times9$)

Per non appesantire troppo l'esecuzione, la griglia usa $N=100$ passi di Trotter e 8 punti di $t$ (invece dei 200/25 usati altrove): sufficiente per farsi un'idea delle forme, non per cifre decimali esatte a $t$ grande.

In [ ]:
t_grid = np.linspace(0.3, 10, 8)
t_grid_fine = np.linspace(0, 10, 200)
N_griglia = 100

risultati, riferimento = {}, {}
for i in sites:
    for alpha in comps:
        for j in sites:
            for beta in comps:
                key = (i, alpha, j, beta)
                risultati[key] = np.array([
                    correlator_from_circuit(i, alpha, j, beta, t, N_griglia, J, Jp, b, D, psi0)
                    for t in t_grid])
                riferimento[key] = classical_exact(i, alpha, j, beta, t_grid_fine, J, Jp, b, D, psi0, H) \
                    if np.isscalar(t_grid_fine) else np.array([
                    classical_exact(i, alpha, j, beta, t, J, Jp, b, D, psi0, H) for t in t_grid_fine])

print(f"Calcolate {len(risultati)} serie temporali (8 punti ciascuna via circuito).")

In [ ]:
fig, axes = plt.subplots(9, 9, figsize=(11, 11), dpi=70, sharex=True)
combo_list = list(risultati.keys())
for ax, key in zip(axes.flat, combo_list):
    i, alpha, j, beta = key
    ref = riferimento[key]
    ax.plot(t_grid_fine, ref.real, color="#0F6E56", lw=0.8)
    ax.plot(t_grid_fine, ref.imag, color="#A4306B", lw=0.8)
    ax.plot(t_grid, risultati[key].real, "o", ms=1.6, color="#0F6E56")
    ax.plot(t_grid, risultati[key].imag, "o", ms=1.6, color="#A4306B")
    ax.set_title(f"{i}{j}:{alpha}{beta}", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

**Lettura della griglia.** La riga/colonna del sito 3 mostra i quattro pannelli piatti a zero
($3x{:}3z$, $3z{:}3x$, $3y{:}3z$, $3z{:}3y$) — lo zero strutturale. Il pannello $33{:}zz$ è quasi
costante e vicino al valore massimo — l'autocorrelazione quasi congelata già notata nello scan a
$t$ fisso. I pannelli con più oscillazioni visibili (es. $11{:}yy$, $12{:}yy$) sono i correlatori
più "ricchi" multi-modali.

## 5. Selettore per ispezione singola

In [ ]:
def plot_correlatore(i, alpha, j, beta):
    key = (i, alpha, j, beta)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(t_grid_fine, riferimento[key].real, color="#0F6E56", lw=1.6, label="classico (esatto)")
    axes[0].plot(t_grid, risultati[key].real, "o", ms=5, color="#993C1D", label="circuito")
    axes[0].set_xlabel("t"); axes[0].set_ylabel(f"Re $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$"); axes[0].legend(fontsize=8)
    axes[1].plot(t_grid_fine, riferimento[key].imag, color="#0F6E56", lw=1.6, label="classico (esatto)")
    axes[1].plot(t_grid, risultati[key].imag, "o", ms=5, color="#993C1D", label="circuito")
    axes[1].set_xlabel("t"); axes[1].set_ylabel(f"Im $C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$"); axes[1].legend(fontsize=8)
    plt.suptitle(f"$C_{{{i},{j}}}^{{{alpha}{beta}}}(t)$")
    plt.tight_layout(); plt.show()

# Nota: niente ipywidgets qui (bug noto di rendering in questo ambiente, vedi
# quantum_simulation_trimero_anello_trotter.ipynb) -- selezione manuale della combinazione.
plot_correlatore(1, "y", 1, "y")

In [ ]:
plot_correlatore(3, "z", 3, "z")  # l'autocorrelazione quasi congelata

## 6. Analisi: escursione, simmetria, spettro

In [ ]:
righe_an = []
for key, arr in risultati.items():
    i, alpha, j, beta = key
    righe_an.append({"i": i, "alpha": alpha, "j": j, "beta": beta,
                      "escursione": np.max(np.abs(arr)) - np.min(np.abs(arr)),
                      "autocorr": i == j})
df_an = pd.DataFrame(righe_an)
print("Le 5 combinazioni con escursione maggiore:")
print(df_an.sort_values("escursione", ascending=False).head(5)
      [["i","alpha","j","beta","escursione"]].to_string(index=False))
print("\nLe 5 con escursione minore (esclusi i quattro zeri strutturali):")
non_zero = df_an[df_an["escursione"] > 1e-6]
print(non_zero.sort_values("escursione").head(5)
      [["i","alpha","j","beta","escursione"]].to_string(index=False))

### Simmetria indotta da $U_\text{anello}$: $C_{11}^{\alpha\beta}=\eta_\alpha\eta_\beta\,C_{22}^{\alpha\beta}$

In [ ]:
ETA = {"x": -1, "y": -1, "z": +1}
print("Confronto C_11 vs C_22 (dal circuito), rapporto atteso eta_alpha*eta_beta:\n")
for alpha in comps:
    for beta in comps:
        c11 = risultati[(1, alpha, 1, beta)]
        c22 = risultati[(2, alpha, 2, beta)]
        mask = np.abs(c22) > 1e-3
        if not np.any(mask):
            continue
        rapporto = np.mean((c11[mask] / c22[mask]).real)
        print(f"  ({alpha},{beta}): rapporto misurato={rapporto:+.2f}   atteso={ETA[alpha]*ETA[beta]:+d}")

### Lo spettro dietro due casi opposti

$C(t)=\sum_k e^{i(E_0-E_k)t}\,a_kb_k$ (8 termini, uno per autostato). Un correlatore "ricco" ha più
pesi $|a_kb_k|$ confrontabili; uno "piatto" ne ha uno dominante.

In [ ]:
def pesi_spettrali(i, alpha, j, beta):
    A, B = site_op(i, alpha), site_op(j, beta)
    Ev, Vm = np.linalg.eigh(H)
    a = Vm.conj().T @ (A @ psi0)
    bvec = Vm.conj().T @ (B @ psi0)
    return Ev - Ev[0], np.abs(np.conj(a) * bvec)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (i, alpha, j, beta, titolo) in zip(axes, [
    (1, "y", 1, "y", "ricco: $C_{11}^{yy}$"),
    (3, "z", 3, "z", "piatto: $C_{33}^{zz}$"),
]):
    dE, pesi = pesi_spettrali(i, alpha, j, beta)
    pesi_norm = pesi / pesi.max()
    colori = ["#5C7288" if k == 0 else "#993C1D" for k in range(8)]
    ax.bar([f"k={k}" for k in range(8)], pesi_norm, color=colori)
    ax.set_title(titolo); ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", labelsize=7)
plt.tight_layout()
plt.show()

**Discussione.** Il caso "ricco" ($C_{11}^{yy}$) ha più modi con peso comparabile — la
sovrapposizione produce il segnale multi-modale visto nella griglia. Il caso "piatto"
($C_{33}^{zz}$) ha un solo modo dominante ($k=0$, il termine costante): coerente con
l'autocorrelazione quasi congelata osservata.

## 7. Confronto con la validazione a shot finiti (risultati precomputati)

Lo scan a shot finiti su tutte le 81 combinazioni richiede ~4-5 minuti (`AerSimulator`, 8192 shot per esecuzione, due esecuzioni per combinazione): qui si riusano i risultati già calcolati e salvati da `scan81_trimero_anello.py`, invece di ripetere il calcolo in questo notebook.

In [ ]:
d = np.load("scan81_results.npz")
print(f"Zeri strutturali, |C| massimo: statevector={float(np.abs(d['M_sv_re']+1j*d['M_sv_im'])[d['is_zero']].max()):.2e}  "
      f"shots={float(np.abs(d['M_sh_re']+1j*d['M_sh_im'])[d['is_zero']].max()):.2e}")
print(f"Errore di Trotter (statevector vs esatto), su tutte le 81: "
      f"media={float(d['err_trotter_mean']):.2e}  max={float(d['err_trotter_max']):.2e}")
print(f"Errore statistico (shots vs statevector), su tutte le 81: "
      f"media={float(d['err_shot_mean']):.2e}  max={float(d['err_shot_max']):.2e}")

## 8. Riepilogo

- Tutte le 81 combinazioni misurate direttamente via circuito (statevector, $N=200$), nessuna
  dedotta per simmetria — validazione: residuo uniforme $\sim10^{-4}$--$10^{-3}$, coerente con
  solo errore di Trotter.
- Vista d'insieme ($9\times9$, $N=100$, 8 punti di $t$): la riga/colonna del sito 3 mostra
  visivamente i quattro zeri strutturali; $C_{33}^{zz}$ è quasi congelato.
- Relazione di simmetria $C_{11}^{\alpha\beta}=\eta_\alpha\eta_\beta\,C_{22}^{\alpha\beta}$
  confermata via circuito per tutte le 9 coppie $(\alpha,\beta)$.
- Lo spettro spiega la distinzione ricco/piatto: più modi comparabili vs un modo dominante.
- Il confronto con lo scan a shot finiti (precomputato) conferma: errore statistico dominante
  su quello di Trotter di oltre un ordine di grandezza a $8192$ shot.

Questo chiude, per il trimero anello, il mirror dei tre notebook della fase correlazioni del
dimero (`correlazioni_dimero_simmetria_U.ipynb`, `correlazioni_dimero_esplorazione.ipynb`,
`circuito_correlazioni_tutte.ipynb`).